# 🎬 Bir Film Gişede Tutar mı? — TMDB 5000 Movies Analizi

**Bootcamp Final Projesi**

Avatar 237 milyon dolarlık bütçeyle çekildi ve 2.78 milyar dolar hasılat yaptı.
Peki bu sonucu film **vizyona girmeden önce** tahmin edebilir miydik?

Bu notebook'ta:

1. Veriyi tanıyor, içindeki gizli sorunları buluyoruz
2. Temizlik ve ön işleme yapıyoruz
3. Grafiklerle hangi filmlerin gerçekten kâr ettiğine bakıyoruz
4. Sadece **çıkış öncesi bilinebilen** bilgilerle bir tahmin modeli kuruyoruz
5. Sonuçları yorumluyoruz

Veri seti: [Kaggle — TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata)

## 0. Kütüphaneler ve Veri

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

Aşağıdaki hücreyi çalıştırıp `tmdb_5000_movies.csv` dosyasını seç.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv("tmdb_5000_movies.csv")
df.shape

---
## 1. Veri Setini Tanıma

TMDB 5000 Movies, The Movie Database'den derlenmiş yaklaşık 4800 filmlik bir veri seti.
Her satır bir film; bütçe, hasılat, tür, süre, çıkış tarihi ve puan bilgilerini içeriyor.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df[["budget", "revenue", "runtime", "vote_average", "vote_count"]].describe()

### 1.1 Eksik Değerler

İlk bakışta veri temiz görünüyor — sadece birkaç kolonda eksik var.

In [ ]:
df.isnull().sum()

In [ ]:
eksik = df.isnull().sum() / len(df) * 100
eksik = eksik[eksik > 0].sort_values()

plt.figure(figsize=(8, 4))
plt.barh(eksik.index, eksik.values, color="#e07a5f")
plt.xlabel("Eksik veri yüzdesi (%)")
plt.title("Kolonlara Göre Eksik Veri Oranı")
plt.savefig("01_eksik_veri.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.2 Asıl Sorun: Sıfırlar

`isnull()` bize yalan söylüyor. `budget` ve `revenue` kolonlarında **hiç eksik değer yok**
gibi görünüyor — çünkü eksik veriler `NaN` yerine **0** olarak kaydedilmiş.

Bir filmin bütçesinin gerçekten 0 dolar olması mümkün değil. Bu satırlar "bilinmiyor" demek.

In [ ]:
print("budget  = 0 olan film sayisi:", (df["budget"] == 0).sum())
print("revenue = 0 olan film sayisi:", (df["revenue"] == 0).sum())

In [ ]:
df[df["budget"] == 0][["title", "budget", "revenue", "vote_count"]].head()

In [ ]:
# Sifirlar hesaba katilinca ortalama butce ne kadar degisiyor?
print("Sifirlar dahil ortalama butce :", df["budget"].mean())
print("Sifirlar haric ortalama butce :", df[df["budget"] > 0]["budget"].mean())

Aradaki fark **%28**. Bu tuzağı fark etmezsek bundan sonraki her hesabımız yanlış olur.

**Ders:** `isnull()` bir başlangıç noktasıdır, bitiş noktası değil. Sayısal kolonlarda
sıfırların da eksik veri olabileceğini kontrol etmek gerekiyor.

---
## 2. Veri Ön İşleme

Şimdi veriyi analize hazır hale getiriyoruz.

### Hedef Değişken: `hit`

Bir filme "başarılı" demek için hasılatının bütçesinin **en az 2 katı** olmasını şart koşuyoruz:

```
hit = 1   eğer  revenue / budget >= 2
hit = 0   diğer durumlarda
```

**Neden 2 kat?** Çünkü `budget` sadece yapım maliyetini içeriyor; pazarlama ve dağıtım
masrafları bunun içinde yok. Sektörde kabaca "bir film başabaş noktasını bütçesinin
iki katında geçer" kuralı kullanılır. Yani `revenue > budget` olması kâr ettiği anlamına gelmez.

**Adım 1:** Sadece vizyona girmiş filmleri alalım.

In [ ]:
veri = df[df["status"] == "Released"].copy()
print(len(veri), "film kaldi")

**Adım 2:** Sıfırları `NaN` (boş) yapalım — çünkü aslında bilinmeyen değerler.

In [ ]:
veri["budget"] = veri["budget"].replace(0, np.nan)
veri["revenue"] = veri["revenue"].replace(0, np.nan)

**Adım 3:** Tarihten yıl ve ay bilgisini çıkaralım.

In [ ]:
veri["release_date"] = pd.to_datetime(veri["release_date"], errors="coerce")
veri["yil"] = veri["release_date"].dt.year
veri["ay"] = veri["release_date"].dt.month

veri[["title", "release_date", "yil", "ay"]].head()

**Adım 4:** `genres` kolonu metin olarak duruyor, listeye çevirelim.

Kolonun içi şöyle görünüyor:

In [ ]:
veri["genres"].iloc[0]

In [ ]:
import ast

def tur_listesi(metin):
    turler = ast.literal_eval(metin)      # metni Python listesine cevirir
    return [t["name"] for t in turler]    # sadece isimleri al

veri["turler"] = veri["genres"].apply(tur_listesi)
veri["sirket_sayisi"] = veri["production_companies"].apply(lambda m: len(ast.literal_eval(m)))
veri["ingilizce"] = (veri["original_language"] == "en").astype(int)

veri[["title", "turler", "sirket_sayisi", "ingilizce"]].head()

**Adım 5:** Hedef değişkenimizi oluşturalım.

In [ ]:
veri["roi"] = veri["revenue"] / veri["budget"]
veri["hit"] = (veri["roi"] >= 2).astype(int)

veri[["title", "budget", "revenue", "roi", "hit"]].head()

**Adım 6:** Bütçesi veya hasılatı bilinmeyen filmleri çıkaralım — bunlar için ROI hesaplayamayız.

Ayrıca çok küçük bütçeler (100 dolar gibi) ROI'yi anlamsız şekilde şişiriyor,
bu yüzden 1.000 doların altını da eliyoruz.

In [ ]:
analiz = veri[(veri["budget"] >= 1000) & (veri["revenue"] > 0)].copy()

print("Ham veri      :", len(df), "film")
print("Analiz seti   :", len(analiz), "film")
print()
print("Hit orani     :", round(analiz["hit"].mean(), 3))
print(analiz["hit"].value_counts())

Veri setinin üçte birini kaybettik (4803 → 3215). Bu bir **hayatta kalma yanlılığı**
yaratıyor: bütçe/hasılat bilgisi genelde büyük stüdyo filmleri için kayıtlı, küçük
bağımsız yapımlar veri setinden düşüyor. Sonuçları yorumlarken bunu unutmayacağız.

İyi haber: sınıf dağılımımız dengeli (%56 hit / %44 başarısız), bu da doğruluk oranını
anlamlı bir metrik yapıyor.

---
## 3. Veri Analizi ve Görselleştirme

### 3.1 Veri Seti Hangi Yıllara Ait?

In [ ]:
yillik = analiz["yil"].value_counts().sort_index()

plt.figure(figsize=(10, 4))
plt.plot(yillik.index, yillik.values, color="#1d3557")
plt.fill_between(yillik.index, yillik.values, color="#3d5a80", alpha=0.6)
plt.xlabel("Çıkış yılı")
plt.ylabel("Film sayısı")
plt.title("Yıllara Göre Film Sayısı")
plt.savefig("02_yillara_gore_film.png", dpi=150, bbox_inches="tight")
plt.show()

print("2000 oncesi :", (analiz["yil"] < 2000).sum(), "film")
print("2000 sonrasi:", (analiz["yil"] >= 2000).sum(), "film")

Filmlerin %71'i 2000 sonrası. Bu iyi haber: eski filmlerin enflasyona göre düzeltilmemiş
bütçeleri analizi çok fazla bozmayacak.

### 3.2 Bütçe ve Hasılat Dağılımı — Neden Logaritma Alıyoruz?

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(analiz["budget"] / 1000000, bins=50, color="#3d5a80")
plt.xlabel("Bütçe (milyon $)")
plt.ylabel("Film sayısı")
plt.title("Ham Dağılım")

plt.subplot(1, 2, 2)
plt.hist(np.log10(analiz["budget"]), bins=50, color="#3d5a80")
plt.xlabel("log10(bütçe)")
plt.ylabel("Film sayısı")
plt.title("Logaritmik Dağılım")

plt.savefig("03_dagilimlar_log.png", dpi=150, bbox_inches="tight")
plt.show()

print("En dusuk butce :", analiz["budget"].min())
print("En yuksek butce:", analiz["budget"].max())

Bütçeler 7 bin dolardan 380 milyon dolara uzanıyor — arada 54 binden fazla kat var.
Ham ölçekte bütün filmler tek bir sütuna sıkışıyor ve hiçbir şey göremiyoruz.
Logaritma tabloyu açıyor. Bu yüzden modelde de bütçenin logaritmasını kullanacağız.

### 3.3 Bütçe ile Hasılat Arasındaki İlişki

In [ ]:
plt.figure(figsize=(8, 6))
renkler = analiz["hit"].map({1: "#2a9d8f", 0: "#e76f51"})
plt.scatter(analiz["budget"], analiz["revenue"], s=12, alpha=0.3, c=renkler)

# Basabas ve 2x cizgileri
x = [1000, 400000000]
plt.plot(x, x, "--", color="gray", label="Hasılat = Bütçe")
plt.plot(x, [2 * 1000, 2 * 400000000], "--", color="black", label="Hasılat = 2 × Bütçe")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Bütçe ($)")
plt.ylabel("Hasılat ($)")
plt.title("Bütçe vs Hasılat — Yeşil Noktalar 'hit' Filmler")
plt.legend()
plt.savefig("04_butce_vs_hasilat.png", dpi=150, bbox_inches="tight")
plt.show()

print("Korelasyon:", round(np.log10(analiz["budget"]).corr(np.log10(analiz["revenue"])), 3))

Güçlü bir ilişki var (0.601): büyük bütçe genelde büyük hasılat getiriyor.
Ama dikkat — bu **kâr** demek değil. Şimdi ona bakalım.

### 3.4 Bütçe Büyüklüğü Başarıyı Nasıl Etkiliyor?

Filmleri bütçelerine göre 5 eşit gruba bölüp her grubun başarı oranına bakalım.
Buradaki sonuç ileride model bölümünü anlamak için önemli olacak.

In [ ]:
analiz["butce_grubu"] = pd.qcut(analiz["budget"], 5,
                                labels=["8M$ altı", "8-20M$", "20-35M$",
                                        "35-65M$", "65M$ üstü"])

grup_ozeti = analiz.groupby("butce_grubu", observed=True)[["budget", "roi"]].median()
grup_ozeti["hit_orani"] = analiz.groupby("butce_grubu", observed=True)["hit"].mean()
grup_ozeti

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.bar(grup_ozeti.index, grup_ozeti["hit_orani"] * 100, color="#2a9d8f")
plt.axhline(analiz["hit"].mean() * 100, color="black", linestyle="--",
            label="Tüm filmlerin ortalaması (%56)")
plt.ylabel("Hit oranı (%)")
plt.xlabel("Yapım bütçesi")
plt.title("Bütçeye Göre Gişe Başarısı")
plt.xticks(fontsize=9)
plt.legend(fontsize=9)

plt.subplot(1, 2, 2)
plt.bar(grup_ozeti.index, grup_ozeti["roi"], color="#3d5a80")
plt.axhline(2, color="black", linestyle="--", label="Başabaş sınırı (2x)")
plt.legend(fontsize=9)
plt.ylabel("Medyan getiri (hasılat / bütçe)")
plt.xlabel("Yapım bütçesi")
plt.title("Bütçeye Göre Yatırım Getirisi")
plt.xticks(fontsize=9)

plt.savefig("05_butce_gruplari.png", dpi=150, bbox_inches="tight")
plt.show()

İlişki düz bir çizgi değil, **U şeklinde.** En düşük bütçeli filmlerin %66'sı hit oluyor,
orta grupta bu oran %48'e düşüyor, sonra en yüksek bütçelilerde tekrar %60'a çıkıyor.

Yani devasa yapımlar aslında oldukça güvenli bir bahis. Riskli olan, ortada sıkışan filmler.

Bu bulguyu aklımızda tutalım: **düz çizgi çizen bir model bu şekli yakalayamaz.**

### 3.5 Hangi Türler Kâr Ediyor?

In [ ]:
# Her filmi turlerine gore ayri satirlara bolelim
tur_bazli = analiz.explode("turler")

tur_ozeti = tur_bazli.groupby("turler")[["roi", "budget"]].median()
tur_ozeti["hit_orani"] = tur_bazli.groupby("turler")["hit"].mean()
tur_ozeti["film_sayisi"] = tur_bazli.groupby("turler")["title"].count()

# En az 30 filmi olan turleri tutalim, digerlerinde sayilar guvenilir olmaz
tur_ozeti = tur_ozeti[tur_ozeti["film_sayisi"] >= 30].sort_values("roi", ascending=False)
tur_ozeti

In [ ]:
plt.figure(figsize=(9, 7))
renkler = ["#2a9d8f" if x >= 2 else "#e76f51" for x in tur_ozeti["roi"]]
plt.barh(tur_ozeti.index, tur_ozeti["roi"], color=renkler)
plt.axvline(2, color="black", linestyle="--", label="Hit sınırı (2x)")
plt.xlabel("Medyan ROI (hasılat / bütçe)")
plt.title("Türlere Göre Medyan Yatırım Getirisi")
plt.gca().invert_yaxis()
plt.legend()
plt.savefig("06_tur_roi.png", dpi=150, bbox_inches="tight")
plt.show()

**Horror türü öne çıkıyor:** 329 filmle güvenilir bir örneklem, medyan bütçesi düşük ve
**hit oranı %67 ile listenin en yükseği.** Korku filmleri ucuza çekiliyor ve tutarlı
şekilde para kazandırıyor.

Diğer uçta `Western` ve `History` türleri medyan olarak başabaş noktasını bile geçemiyor.

### 3.6 Çıkış Ayının Etkisi Var mı?

In [ ]:
ay_ozeti = analiz.groupby("ay")["hit"].mean()
ay_isimleri = ["Oca", "Şub", "Mar", "Nis", "May", "Haz",
               "Tem", "Ağu", "Eyl", "Eki", "Kas", "Ara"]

# En iyi ve en kotu ayi farkli renkte gosterelim
renkler = []
for oran in ay_ozeti.values:
    if oran == ay_ozeti.max():
        renkler.append("#2a9d8f")
    elif oran == ay_ozeti.min():
        renkler.append("#e76f51")
    else:
        renkler.append("#a8c5c1")

plt.figure(figsize=(10, 4.5))
plt.bar(ay_isimleri, ay_ozeti.values * 100, color=renkler)
plt.axhline(analiz["hit"].mean() * 100, color="black", linestyle="--", label="Genel ortalama")
plt.ylabel("Hit oranı (%)")
plt.xlabel("Çıkış ayı")
plt.title("Çıkış Ayına Göre Gişe Başarısı")
plt.legend()
plt.savefig("07_ay_hit_orani.png", dpi=150, bbox_inches="tight")
plt.show()

print(round(ay_ozeti * 100, 1))

Haziran (%67) ile eylül (%43) arasında **24 puanlık** fark var. Yaz tatili ve yılbaşı
sezonu beklendiği gibi güçlü.

İlginç olan şu: eylül veri setinde **en çok film çıkan ay** (383 film) ama aynı zamanda
en düşük başarı oranına sahip. Sektörde bunun bir adı var — stüdyoların güvenmediği
filmleri "boşalttığı" ay.

### 3.7 ROI'nin Uç Değer Problemi

Devam etmeden önce önemli bir tuzağı görelim. Bu notebook boyunca neden hep
**ortalama değil medyan** kullandığımızın sebebi burada.

In [ ]:
print("ROI ortalamasi :", round(analiz["roi"].mean(), 2))
print("ROI medyani    :", round(analiz["roi"].median(), 2))

In [ ]:
analiz.nlargest(5, "roi")[["title", "budget", "revenue", "roi"]]

`Paranormal Activity` 15 bin dolara çekilip 193 milyon hasılat yapmış — **12.890 kat** getiri.
Tek başına bu film, tüm veri setinin ortalama ROI'sini 11 kata çıkarıyor; oysa tipik bir
filmin getirisi 2.3 kat.

Uç değerlerin olduğu bir dağılımda ortalama, hiç kimseyi temsil etmeyen bir sayıdır.

### 3.8 İyi Film mi, Kârlı Film mi?

In [ ]:
puan_gruplari = pd.cut(analiz["vote_average"],
                       bins=[0, 5, 6, 6.5, 7, 7.5, 10],
                       labels=["5 altı", "5-6", "6-6.5", "6.5-7", "7-7.5", "7.5 üstü"])

puan_ozeti = analiz.groupby(puan_gruplari, observed=True)[["roi"]].median()
puan_ozeti["hit_orani"] = analiz.groupby(puan_gruplari, observed=True)["hit"].mean()
puan_ozeti

In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.bar(puan_ozeti.index.astype(str), puan_ozeti["roi"], color="#3d5a80")
plt.axhline(2, color="black", linestyle="--")
plt.xlabel("TMDB puanı")
plt.ylabel("Medyan ROI")
plt.title("Puana Göre Getiri")
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.bar(puan_ozeti.index.astype(str), puan_ozeti["hit_orani"] * 100, color="#2a9d8f")
plt.xlabel("TMDB puanı")
plt.ylabel("Hit oranı (%)")
plt.title("Puana Göre Hit Oranı")
plt.xticks(rotation=45)

plt.savefig("08_puan_vs_roi.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Pearson korelasyonu :", round(analiz["vote_average"].corr(analiz["roi"]), 3))
print("Spearman korelasyonu:", round(analiz["vote_average"].corr(analiz["roi"], method="spearman"), 3))

Burada güzel bir istatistik dersi var. **Pearson korelasyonu 0.000 çıkıyor** — sanki puan
ile kârlılık arasında hiçbir ilişki yokmuş gibi. Ama bu yanlış bir sonuç: Pearson uç
değerlere aşırı duyarlı ve `Paranormal Activity`'nin 12.890'lık ROI'si katsayıyı tek
başına eziyor.

**Spearman korelasyonu** ham değerler yerine sıralamalarla çalıştığı için bu tuzağa
düşmüyor ve **0.335** veriyor: orta güçte, pozitif, anlamlı bir ilişki. Grafik de bunu
doğruluyor — 7.5+ puan alan filmlerin %82'si hit olurken, 5 altı puan alanların sadece
%31'i hit oluyor.

Yani iyi film yapmak gerçekten para kazandırıyor. Ama şunu unutmayalım:
**puan bilgisi film çıkmadan önce elimizde olmuyor.**

### 3.9 Sayısal Değişkenler Arası Korelasyon

In [ ]:
sayisal = analiz[["budget", "revenue", "runtime", "popularity",
                  "vote_average", "vote_count", "sirket_sayisi", "roi"]]

plt.figure(figsize=(8.5, 7))
sns.heatmap(sayisal.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Korelasyon Matrisi")
plt.savefig("09_korelasyon.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Makine Öğrenmesi Modeli

### 4.1 En Kritik Karar: Hangi Özellikleri Kullanacağız?

Elimizde `popularity`, `vote_count`, `vote_average` gibi güçlü değişkenler var.
Bunları modele koysak doğruluk oranı ciddi şekilde yükselirdi. **Ama koymayacağız.**

Sebep: bu üç değişken de film **vizyona girdikten sonra** oluşuyor. Bir filmin kaç
kişi tarafından oylandığını bilmek, o filmin çok izlendiğini bilmek demektir — yani
cevabı soruyla birlikte modele vermek olur. Buna **veri sızıntısı (data leakage)** denir.

Model kâğıt üzerinde harika görünür ama gerçek kullanım anında — yani "bu senaryoyu
çekelim mi?" kararı verilirken — elimizde bu bilgilerin hiçbiri olmaz.

| Kullanacağız ✅ | Kullanmayacağız ❌ | Neden |
|---|---|---|
| `budget` (logaritmalı) | `revenue` | Hedefin kendisi |
| `runtime` | `popularity` | Çıkış sonrası oluşur |
| `yil`, `ay` | `vote_count` | Çıkış sonrası oluşur |
| `sirket_sayisi` | `vote_average` | Çıkış sonrası oluşur |
| `ingilizce` | | |
| Türler (0/1 kolonları) | | |

Önce türleri sayıya çevirelim. Her tür için bir kolon açıp 0 veya 1 yazacağız.

In [ ]:
tum_turler = analiz["turler"].explode().dropna().unique()
print("Toplam tur sayisi:", len(tum_turler))

for tur in tum_turler:
    analiz["tur_" + tur] = analiz["turler"].apply(lambda liste: 1 if tur in liste else 0)

analiz[["title", "turler", "tur_Action", "tur_Drama", "tur_Horror"]].head()

In [ ]:
# Eksik sureleri medyan ile dolduralim (sadece birkac satir)
analiz["runtime"] = analiz["runtime"].fillna(analiz["runtime"].median())
analiz = analiz.dropna(subset=["yil"])

# Butcenin logaritmasini aliyoruz - dagilimi cok carpikti
analiz["log_butce"] = np.log10(analiz["budget"])

ozellikler = ["log_butce", "runtime", "yil", "ay", "sirket_sayisi", "ingilizce"]
ozellikler = ozellikler + ["tur_" + t for t in tum_turler]

X = analiz[ozellikler]
y = analiz["hit"]

print("Ozellik sayisi:", len(ozellikler))
print("Film sayisi   :", len(X))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

print("Egitim seti:", len(X_train), "film")
print("Test seti  :", len(X_test), "film")

### 4.2 Baseline — Yenmemiz Gereken Çizgi

Model kurmadan önce en basit tahminin ne kadar başarılı olduğunu bilmeliyiz:
**her filme "hit olur" de.** Modelimiz bunu geçemezse hiçbir işe yaramıyor demektir.

In [ ]:
print("Baseline dogruluk:", round(y_test.mean(), 3))

### 4.3 Model 1 — Lojistik Regresyon

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Lojistik regresyon icin degerleri ayni olcege getirmemiz gerekiyor
olcekleyici = StandardScaler()
X_train_olcekli = olcekleyici.fit_transform(X_train)
X_test_olcekli = olcekleyici.transform(X_test)

logreg = LogisticRegression(max_iter=2000)
logreg.fit(X_train_olcekli, y_train)

logreg_tahmin = logreg.predict(X_test_olcekli)
logreg_olasilik = logreg.predict_proba(X_test_olcekli)[:, 1]

print("Dogruluk :", round(accuracy_score(y_test, logreg_tahmin), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, logreg_olasilik), 3))
print()
print(classification_report(y_test, logreg_tahmin, target_names=["Başarısız", "Hit"]))

### 4.4 Model 2 — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, max_depth=12,
                            min_samples_leaf=5, random_state=42)
rf.fit(X_train, y_train)

rf_tahmin = rf.predict(X_test)
rf_olasilik = rf.predict_proba(X_test)[:, 1]

print("Dogruluk :", round(accuracy_score(y_test, rf_tahmin), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, rf_olasilik), 3))
print()
print(classification_report(y_test, rf_tahmin, target_names=["Başarısız", "Hit"]))

### 4.5 Çapraz Doğrulama

Yukarıdaki skorlar tek bir rastgele test setinden geliyor. Şanslı ya da şanssız bir
bölünmeye denk gelmiş olabiliriz. 5 katlı çapraz doğrulama ile kontrol edelim.

⚠️ **Burada dikkat edilmesi gereken bir nokta var.** `cv=5` yazarsak scikit-learn veriyi
**karıştırmadan** böler. Bu veri setinde bu ciddi bir hata olur, çünkü CSV kabaca bütçeye
göre sıralı: başta Avatar gibi 200 milyon dolarlık yapımlar, sonlarda `Paranormal Activity`
gibi 15 bin dolarlık filmler var.

Karıştırmadan bölersek her kat tamamen farklı bir bütçe aralığından oluşur.
`shuffle=True` bunu çözüyor.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# YANLIS - veriyi karistirmadan boluyor
skorlar = cross_val_score(rf, X, y, cv=5)
print("Karistirmadan :", round(skorlar.mean(), 3))

# DOGRU - once karistirip sonra boluyor
kat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skorlar = cross_val_score(rf, X, y, cv=kat)
print("Karistirarak  :", round(skorlar.mean(), 3), "+/-", round(skorlar.std(), 3))
print(skorlar.round(3))

Fark çok büyük: karıştırmadan bölünce Random Forest %55.4 görünüyor — yani baseline'ın
bile altında. Doğru yöntemle %62.0'ye çıkıyor.

Bu aynı zamanda test setindeki %65.9'un biraz iyimser olduğunu gösteriyor.
**Dürüst rakam çapraz doğrulamadan gelen değer.**

### 4.6 Sonuçların Karşılaştırması

In [ ]:
sonuclar = pd.DataFrame({
    "Model": ["Baseline (hep 'hit')", "Lojistik Regresyon", "Random Forest"],
    "Dogruluk": [round(y_test.mean(), 3),
                 round(accuracy_score(y_test, logreg_tahmin), 3),
                 round(accuracy_score(y_test, rf_tahmin), 3)],
    "ROC-AUC": [0.5,
                round(roc_auc_score(y_test, logreg_olasilik), 3),
                round(roc_auc_score(y_test, rf_olasilik), 3)],
})
sonuclar

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
matris = confusion_matrix(y_test, rf_tahmin)
sns.heatmap(matris, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Başarısız", "Hit"], yticklabels=["Başarısız", "Hit"])
plt.xlabel("Tahmin")
plt.ylabel("Gerçek")
plt.title("Karışıklık Matrisi — Random Forest")

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_test, logreg_olasilik)
plt.plot(fpr, tpr, color="#3d5a80", label="Lojistik Regresyon")
fpr, tpr, _ = roc_curve(y_test, rf_olasilik)
plt.plot(fpr, tpr, color="#2a9d8f", label="Random Forest")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Rastgele tahmin")
plt.xlabel("Yanlış pozitif oranı")
plt.ylabel("Doğru pozitif oranı")
plt.title("ROC Eğrisi")
plt.legend()

plt.savefig("10_model_sonuclari.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.7 Hangi Özellik Ne Kadar Önemli?

In [ ]:
onem = pd.Series(rf.feature_importances_, index=ozellikler).sort_values(ascending=False)

plt.figure(figsize=(9, 6))
plt.barh(onem.head(15).index, onem.head(15).values, color="#3d5a80")
plt.xlabel("Önem skoru")
plt.title("Random Forest — En Önemli 15 Özellik")
plt.gca().invert_yaxis()
plt.savefig("11_ozellik_onemi.png", dpi=150, bbox_inches="tight")
plt.show()

print(onem.head(6).round(3))

In [ ]:
# Lojistik regresyonun katsayilari yon bilgisi verir (artiriyor mu, azaltiyor mu)
katsayilar = pd.Series(logreg.coef_[0], index=ozellikler).sort_values()

print("Hit olasiligini EN COK DUSURENLER:")
print(katsayilar.head(3).round(3))
print()
print("Hit olasiligini EN COK ARTIRANLAR:")
print(katsayilar.tail(3).round(3))

Burada ilginç bir şey var. `log_butce` Random Forest'ın en önemli iki değişkeninden biri,
ama lojistik regresyonda katsayısı **negatif** — yani "bütçe arttıkça hit olma olasılığı
düşer" diyor.

Oysa 3.4'teki U eğrisi bunun doğru olmadığını göstermişti: en yüksek bütçeli grubun
hit oranı %59.5 ile ortalamanın üzerindeydi.

Çelişki değil, **doğrusal modelin sınırı.** Lojistik regresyon sadece düz çizgi çizebiliyor.
U şeklindeki bir ilişkiye düz çizgi uydurmaya çalışınca aşağı eğimli bir çizgi çıkıyor.
Random Forest ise ağaç tabanlı olduğu için bu şekli öğrenebiliyor —
test setinde **%59.3'e karşı %65.9** farkının asıl sebebi bu.

### 4.8 Modeli Deneyelim: Hayali Filmler

Modeli somut senaryolarda çalıştıralım.

In [ ]:
def tahmin_et(butce, sure, ay, film_turleri, yil=2016, sirket=3):
    # Once tum ozellikleri 0 yapip bos bir satir olusturuyoruz
    satir = pd.DataFrame(0, index=[0], columns=ozellikler)

    satir["log_butce"] = np.log10(butce)
    satir["runtime"] = sure
    satir["yil"] = yil
    satir["ay"] = ay
    satir["sirket_sayisi"] = sirket
    satir["ingilizce"] = 1
    for tur in film_turleri:
        satir["tur_" + tur] = 1

    return rf.predict_proba(satir)[0][1]


print("40M $ gerilim, eylul       :", round(tahmin_et(40000000, 110, 9, ["Thriller"]), 3))
print("40M $ gerilim, haziran     :", round(tahmin_et(40000000, 110, 6, ["Thriller"]), 3))
print("20M $ korku, eylul         :", round(tahmin_et(20000000, 100, 9, ["Horror"]), 3))
print("200M $ aksiyon, haziran    :", round(tahmin_et(200000000, 140, 6, ["Action", "Adventure"]), 3))

Model, 200 milyonluk blockbuster'a 40 milyonluk gerilim filminden **15 puan** daha yüksek
şans veriyor — yani U eğrisinin sağ kolunu gerçekten öğrenmiş.

---
## 5. Sonuç ve Yorum

### Ne Bulduk?

**1. Bütçe ile başarı arasındaki ilişki U şeklinde.** En ucuz filmler (%66) ve en pahalı
filmler (%60) iyi gidiyor, ortada sıkışanlar (%48) batıyor.

**2. Horror en tutarlı tür.** Düşük bütçe, %67 hit oranı.

**3. Çıkış ayı önemli.** Haziran %67, eylül %43. Aradaki fark 24 puan.

**4. İyi film ile kârlı film arasında bağ var** — ama görmek için doğru aracı seçmek
gerekiyor. Pearson 0.000 diyerek bizi yanıltıyordu, Spearman 0.335 ile gerçeği gösterdi.

**5. Model baseline'ı geçiyor ama tahmin gücü sınırlı.** Random Forest çapraz doğrulamada
baseline'ın yaklaşık 6 puan üzerine çıkabiliyor. **Ve bu beklenen bir sonuç.** Gişe başarısı;
oyuncu kadrosu, yönetmen, pazarlama bütçesi, o hafta çıkan rakip filmler ve saf şansın
birleşimiyle belirleniyor. Elimizdeki 25 özellik bunun küçük bir kısmını yakalıyor.

Eğer `popularity` ve `vote_count` değişkenlerini modele koysaydık doğruluk çok daha
yüksek çıkardı — ama o model gerçek hayatta kullanılamazdı. **Yüksek skor her zaman
iyi model demek değildir.**

### Bu Projede Düştüğüm 4 Tuzak

Hepsi sessizdi. Hiçbiri hata vermedi, hepsi güzel bir sayı üretti ve o sayı yanlıştı:

1. `isnull()` "eksik yok" dedi — eksikler 0 olarak saklanıyordu
2. Ortalama ROI 11x dedi — tek bir film yüzünden
3. Pearson korelasyonu 0.000 dedi — gerçek ilişki 0.335'ti
4. Çapraz doğrulama %55.6 dedi — `shuffle=True` eksikti

### Kısıtlar

- **Hayatta kalma yanlılığı:** 4803 filmin sadece 3215'inde bütçe/hasılat vardı
- **Enflasyon düzeltmesi yok:** 1916 ile 2016 dolarları aynı kabul edildi
- **Dil dengesizliği:** Filmlerin %94'ü İngilizce, sonuçlar esasen Hollywood'u anlatıyor
- **`budget` pazarlamayı içermiyor:** 2x kuralı bir yaklaşım, kesin bir muhasebe değil

### Sonraki Adımlar

- Oyuncu ve yönetmen bilgisini eklemek (TMDB credits veri seti)
- Film özetlerinden (`overview`) metin özellikleri çıkarmak
- Bütçeleri enflasyona göre bugünkü değerine çevirmek
- XGBoost gibi daha güçlü modeller denemek

---
## 6. Grafikleri İndir

Grafikler notebook'un yanına PNG olarak kaydedildi. Colab'de sol taraftaki
**klasör simgesine** tıklayıp dosyaları tek tek indirebilirsin.